# Kafka vs Kinesis — When Each Wins

## Mental Model

Kafka and Kinesis both move event streams, but they optimize for different operating models.

- **Kafka** wins when you want infrastructure control, broad ecosystem portability, deep replay patterns, and cross-cloud flexibility.
- **Kinesis** wins when you are AWS-native and want a managed streaming service with minimal broker operations.

Citi framing used in this notebook:

- 6,000+ API endpoints monitored for latency, throughput, and error rate
- alerts escalate through severity tiers
- local Kafka models the on-prem / self-managed event backbone
- Kinesis models AWS-native ingestion for downstream Firehose → S3 style pipelines


In [ ]:
import json
import os
import time
from statistics import quantiles

import boto3
import psycopg2
from botocore.exceptions import ClientError
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

os.environ['AWS_PROFILE'] = 'study'
AWS_PROFILE = 'study'
AWS_REGION = 'us-east-1'
AWS_ACCOUNT = '357811130281'

KAFKA_BOOTSTRAP = 'localhost:9092'
KAFKA_TOPIC = 'citi.comparison'
KINESIS_STREAM = 'citi-comparison'

POSTGRES_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'de_telemetry',
    'user': 'de_admin',
    'password': 'DeAdmin2026!',
}

admin = AdminClient({'bootstrap.servers': KAFKA_BOOTSTRAP})
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
kinesis = session.client('kinesis')

def pg_conn():
    return psycopg2.connect(**POSTGRES_CONFIG)

def p99_ms(values):
    if not values:
        return 0.0
    if len(values) == 1:
        return round(values[0] * 1000, 3)
    return round(quantiles(values, n=100, method='inclusive')[98] * 1000, 3)

def ensure_kafka_topic(topic_name: str, num_partitions: int = 3, replication_factor: int = 1):
    md = admin.list_topics(timeout=10)
    if topic_name in md.topics and not md.topics[topic_name].error:
        return
    futures = admin.create_topics([
        NewTopic(topic_name, num_partitions=num_partitions, replication_factor=replication_factor)
    ])
    for _, fut in futures.items():
        try:
            fut.result()
        except Exception as exc:
            if 'already exists' not in str(exc).lower():
                raise

def stream_exists(stream_name: str) -> bool:
    try:
        kinesis.describe_stream_summary(StreamName=stream_name)
        return True
    except ClientError as exc:
        code = exc.response.get('Error', {}).get('Code', '')
        if code in {'ResourceNotFoundException', '404'}:
            return False
        raise

def wait_for_stream_status(stream_name: str, desired_status: str, timeout_s: int = 180):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            resp = kinesis.describe_stream_summary(StreamName=stream_name)
            status = resp['StreamDescriptionSummary']['StreamStatus']
            if status == desired_status:
                return status
        except ClientError as exc:
            code = exc.response.get('Error', {}).get('Code', '')
            if desired_status == 'DELETED' and code in {'ResourceNotFoundException', '404'}:
                return 'DELETED'
            raise
        time.sleep(2)
    raise TimeoutError(f'Stream {stream_name} did not reach status {desired_status}')

ensure_kafka_topic(KAFKA_TOPIC)
print(f'Kafka bootstrap: {KAFKA_BOOTSTRAP}')
print(f'Kafka topic ready: {KAFKA_TOPIC}')
print(f'AWS profile: {AWS_PROFILE} | region: {AWS_REGION} | account: {AWS_ACCOUNT}')
print(f'Kinesis stream target: {KINESIS_STREAM}')
print(f"PostgreSQL: {POSTGRES_CONFIG['host']}:{POSTGRES_CONFIG['port']}/{POSTGRES_CONFIG['dbname']}")


## Load Citi Alert Sample

We pull 50 alert rows from PostgreSQL and reuse the same payload for both Kafka and Kinesis so the comparison is apples-to-apples.


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT a.alert_id,
                   a.endpoint_id,
                   a.severity,
                   a.message,
                   a.created_at,
                   e.name,
                   e.region,
                   e.status,
                   e.category
            FROM alerts a
            JOIN endpoints e
              ON e.endpoint_id = a.endpoint_id
            ORDER BY a.created_at DESC, a.alert_id DESC
            LIMIT 50
        ''')
        rows = cur.fetchall()

if len(rows) != 50:
    raise RuntimeError(f'Expected 50 rows, got {len(rows)}')

records = []
for row in rows:
    payload = {
        'alert_id': row[0],
        'endpoint_id': row[1],
        'severity': row[2],
        'message': row[3],
        'created_at': row[4].isoformat(),
        'endpoint_name': row[5],
        'region': row[6],
        'endpoint_status': row[7],
        'category': row[8],
        'source': 'citi_telemetry_compare',
    }
    records.append(payload)

print(f'Prepared {len(records)} records for Kafka and Kinesis comparison')
print(json.dumps(records[0], indent=2))


## Kafka — Produce 50 Alerts and Measure Latency

For Kafka, we time each `produce()` enqueue plus the final `flush()` barrier so we get a practical producer-side latency profile.


In [ ]:
kafka_producer = Producer({
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'enable.idempotence': True,
    'acks': 'all',
    'linger.ms': 0,
})

kafka_latencies = []
delivery_reports = []

def kafka_delivery(err, msg):
    if err is not None:
        raise RuntimeError(str(err))
    delivery_reports.append((msg.partition(), msg.offset()))

for payload in records:
    payload_bytes = json.dumps(payload).encode('utf-8')
    start = time.perf_counter()
    kafka_producer.produce(
        KAFKA_TOPIC,
        key=payload['severity'].encode('utf-8'),
        value=payload_bytes,
        callback=kafka_delivery,
    )
    kafka_producer.poll(0)
    kafka_latencies.append(time.perf_counter() - start)

flush_start = time.perf_counter()
remaining = kafka_producer.flush(15)
flush_elapsed = time.perf_counter() - flush_start
if remaining != 0:
    raise RuntimeError(f'Kafka flush incomplete, remaining messages={remaining}')

kafka_latencies.append(flush_elapsed)
print(f'Kafka delivery reports: {len(delivery_reports)}')
print(f'Kafka flush latency: {round(flush_elapsed * 1000, 3)}ms')
print(f'Kafka P99 produce latency: {p99_ms(kafka_latencies)}ms')


## Kinesis — Create Stream, Put 50 Records, Measure Latency, Delete Stream

For Kinesis, we create a single-shard stream, wait until it becomes ACTIVE, put 50 records, and then delete it so the notebook cleans up after itself.


In [ ]:
if stream_exists(KINESIS_STREAM):
    kinesis.delete_stream(StreamName=KINESIS_STREAM, EnforceConsumerDeletion=True)
    wait_for_stream_status(KINESIS_STREAM, 'DELETED', timeout_s=180)

kinesis.create_stream(StreamName=KINESIS_STREAM, ShardCount=1)
wait_for_stream_status(KINESIS_STREAM, 'ACTIVE', timeout_s=180)

kinesis_latencies = []
put_responses = []

try:
    for payload in records:
        payload_bytes = json.dumps(payload).encode('utf-8')
        start = time.perf_counter()
        resp = kinesis.put_record(
            StreamName=KINESIS_STREAM,
            Data=payload_bytes,
            PartitionKey=payload['severity'],
        )
        elapsed = time.perf_counter() - start
        kinesis_latencies.append(elapsed)
        put_responses.append((resp['ShardId'], resp['SequenceNumber']))

    print(f'Kinesis put responses: {len(put_responses)}')
    print(f'Kinesis P99 put latency: {p99_ms(kinesis_latencies)}ms')
finally:
    if stream_exists(KINESIS_STREAM):
        kinesis.delete_stream(StreamName=KINESIS_STREAM, EnforceConsumerDeletion=True)
        wait_for_stream_status(KINESIS_STREAM, 'DELETED', timeout_s=180)
        print(f'Kinesis stream deleted: {KINESIS_STREAM}')


## Decision Matrix

This table is the architectural shorthand you can use in interviews and design reviews.


In [ ]:
decision_rows = [
    ['managed overhead', 'Higher if self-managed; lower with Confluent-managed offerings', 'Low; AWS-managed service'],
    ['ordering guarantee', 'Per partition', 'Per shard'],
    ['replay window', 'Retention-based; can be long and flexible', 'Retention-based; commonly sized by AWS retention settings'],
    ['consumer model', 'Pull-based consumers and consumer groups', 'Shared throughput readers, enhanced fan-out optional'],
    ['pricing model', 'Infra + ops + storage/network; or managed platform pricing', 'Shard / on-demand streaming charges + PUT payload units + retention'],
    ['max throughput', 'Scales by partitions, brokers, and cluster design', 'Scales by shards or on-demand capacity mode'],
    ['multi-cloud', 'Strong fit', 'AWS-centric'],
    ['Citi recommendation', 'Use for on-prem, hybrid, cross-cloud, and deep replay workloads', 'Use for AWS-native ingestion into Firehose -> S3 and managed pipelines'],
]

def format_table(rows, headers):
    str_rows = [[str(x) for x in row] for row in rows]
    widths = [len(str(h)) for h in headers]
    for row in str_rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(value))
    header_line = ' | '.join(str(headers[i]).ljust(widths[i]) for i in range(len(headers)))
    sep_line = '-+-'.join('-' * widths[i] for i in range(len(headers)))
    body = [' | '.join(row[i].ljust(widths[i]) for i in range(len(headers))) for row in str_rows]
    return '\n'.join([header_line, sep_line] + body)

print(format_table(decision_rows, ['dimension', 'kafka', 'kinesis']))


## Migration Pattern — Dual-Write Shim

A common migration move is the **strangler fig pattern**:

- keep the old path alive
- introduce the new path beside it
- dual-write for a controlled transition period
- validate downstream parity
- then retire the old path

The code below shows a reusable dual-write shim that can send the same payload to Kafka and Kinesis during migration.


In [ ]:
class DualWriteShim:
    def __init__(self, kafka_bootstrap: str, kafka_topic: str, kinesis_client, kinesis_stream: str):
        self.kafka_topic = kafka_topic
        self.kinesis_stream = kinesis_stream
        self.kinesis_client = kinesis_client
        self.kafka_producer = Producer({
            'bootstrap.servers': kafka_bootstrap,
            'enable.idempotence': True,
            'acks': 'all',
        })

    def write(self, payload: dict, partition_key: str):
        payload_bytes = json.dumps(payload).encode('utf-8')
        self.kafka_producer.produce(
            self.kafka_topic,
            key=partition_key.encode('utf-8'),
            value=payload_bytes,
        )
        self.kafka_producer.flush(15)
        kinesis_resp = self.kinesis_client.put_record(
            StreamName=self.kinesis_stream,
            Data=payload_bytes,
            PartitionKey=partition_key,
        )
        return {
            'kafka_topic': self.kafka_topic,
            'kinesis_stream': self.kinesis_stream,
            'kinesis_shard_id': kinesis_resp['ShardId'],
            'kinesis_sequence_number': kinesis_resp['SequenceNumber'],
        }

kinesis.create_stream(StreamName=KINESIS_STREAM, ShardCount=1)
wait_for_stream_status(KINESIS_STREAM, 'ACTIVE', timeout_s=180)

try:
    shim = DualWriteShim(KAFKA_BOOTSTRAP, KAFKA_TOPIC, kinesis, KINESIS_STREAM)
    migration_payload = {
        'migration_demo': True,
        'alert_id': records[0]['alert_id'],
        'endpoint_id': records[0]['endpoint_id'],
        'severity': records[0]['severity'],
        'message': records[0]['message'],
        'created_at': records[0]['created_at'],
        'pattern': 'strangler_fig_dual_write',
    }
    shim_result = shim.write(migration_payload, partition_key=migration_payload['severity'])
    print(json.dumps(shim_result, indent=2))
finally:
    if stream_exists(KINESIS_STREAM):
        kinesis.delete_stream(StreamName=KINESIS_STREAM, EnforceConsumerDeletion=True)
        wait_for_stream_status(KINESIS_STREAM, 'DELETED', timeout_s=180)
        print(f'Cleanup delete complete for stream: {KINESIS_STREAM}')


## What Just Happened

You just compared Kafka and Kinesis with the same Citi alert payloads and live write patterns.

- Kafka produced 50 local records against the Citi Kafka stack
- Kinesis created a temporary stream, accepted 50 records, and cleaned itself up
- the decision matrix translated feature differences into architecture language
- the dual-write shim showed how a migration can preserve payload parity during a transition

**Use Kafka when you control infrastructure and need sub-ms latency or cross-cloud portability. Use Kinesis when you are AWS-native and want zero-ops streaming. Citi uses Kafka on-prem; AWS workloads use Kinesis for event ingestion into Firehose → S3.**
